In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import io
from PIL import Image
import numpy as np
import torch

model_params = {
    'x_min': -1.1,
    'x_max': 1.1,
    'y_min': -1.1,
    'y_max': 1.1,
    'u_max': 1.25,
    'radius': 0.5,
    'dt': 0.05,
    'v': 1.,
    'dpi': 128
}

def inv_dynamincs_onestep(s0, param):
    v = param['v']
    dt = param['dt']
    bs = s0.shape[0]
    s, s_prev = torch.zeros(bs, 3), torch.zeros(bs, 3)
    s0 = torch.tensor(s0, dtype=torch.float32)
    s[:,0], s[:,1], s[:,2] = s0[:,0], s0[:,1], s0[:,2]
    s_prev[:,0] = s[:,0] - v*dt*torch.cos(s[:,2])
    s_prev[:,1] = s[:,1] - v*dt*torch.sin(s[:,2])
    s_prev[:,2] = s[:,2]
    return s_prev

def state_to_data(s0, param):
    state_obs, img_obs, state_gt, dones, acs, demos = ([] for _ in range(6))
    
    for i in range(s0.shape[0]):
        s = s0[i]
        ac = 0 * torch.rand(1)
        state_obs.append(s[2].numpy()) # get to observe theta
        state_gt.append(s.numpy()) # gt state
        dones.append(1)
        acs.append(ac)

        dt = param['dt']
        v = param['v']
        center = (0.0, 0.0)

        fig,ax = plt.subplots()
        plt.xlim([-1.1, 1.1]); plt.ylim([-1.1, 1.1])
        plt.axis('off')
        fig.set_size_inches( 1, 1 )
        # Create the circle patch
        circle = patches.Circle(center, param['radius'], 
                                edgecolor=(1,0,0), facecolor='none')
        ax.add_patch(circle)

        plt.quiver(s[0], s[1], dt*v*torch.cos(s[2]), dt*v*torch.sin(s[2]),
                angles='xy', scale_units='xy', minlength=0,width=0.1, 
                scale=0.18,color=(0,0,1), zorder=3)
        plt.scatter(s[0], s[1],s=20, color=(0,0,1), zorder=3)
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=param['dpi'])
        buf.seek(0)

        # Load the buffer content as an RGB image
        img = Image.open(buf).convert('RGB')
        img_array = np.array(img)
        img_obs.append(img_array)
        plt.close()
    demo = {}
    demo['obs'] = {'image': img_obs, 'state': state_obs, 'priv_state': state_gt}
    demo['actions'] = acs
    demo['dones'] = dones

    return demo

In [ ]:
from collections import defaultdict

np_expdim = lambda x: np.expand_dims(x, axis=0)

def single_data_dubins(demos):       
        
    pixel_keys = sorted(['image'])
    state_keys = sorted(['state'])

    traj = demos
    traj_to_pp = {}

    # import pdb; pdb.set_trace()
    for t in range(len(traj["obs"][pixel_keys[0]])):
        transition = defaultdict(np.array)
        for obs_key in pixel_keys:
            transition[obs_key] = traj["obs"][obs_key][t]

        if len(state_keys) != 0:
            curr_obs_state_vec = [traj["obs"][obs_key][t] for obs_key in state_keys]
            transition["state"] = curr_obs_state_vec
            
        transition["privileged_state"] = traj['obs']['priv_state'][t]
        transition["obs_state"] = [np.cos(traj['obs']['state'][t]), 
                                   np.sin(traj['obs']['state'][t])]
        transition["reward"] = np.array(0, dtype=np.float32)
        transition["is_first"] = np.array(t == 0, dtype=np.bool_)
        transition["is_last"] = np.array(traj["dones"][t], dtype=np.bool_)
        transition["is_terminal"] = np.array(traj["dones"][t], dtype=np.bool_)
        transition["discount"] = np.array(1, dtype=np.float32)
        transition["action"] = np.array(traj["actions"][t], dtype=np.float32)

        if t == 0: 
            traj_to_pp = {k:np_expdim(np_expdim(v)) for k,v in transition.items()}
        else: 
            for k,v in traj_to_pp.items():
                traj_to_pp[k] = np.append(v, np_expdim(np_expdim(transition[k])), axis=0)
    return traj_to_pp

In [ ]:
import argparse
import sys
import gym
import numpy as np
import torch

parent_dir = "/home/clown2/Desktop/Work/Courses/EAIS/"
dreamer_dir = "/home/clown2/Desktop/Work/Courses/EAIS/eais_hw2/dreamerv3-torch"
ckpt_path = '/home/clown2/Desktop/Work/Courses/EAIS/logs/dreamer_dubins/best_pretrain_joint_0_12.pt'
policy_path = ("dreamer_l2_lessrand/lcrl/0411/142538/lcrl/dubins-wm/"
                "wm_actor_activation_ReLU_critic_activation_ReLU_game_"
                "gd_steps_1_tau_0.005_training_num_1_buffer_size_40000"
                "_c_net_512_4_a1_512_4_a2_512_4_gamma_0.95/noise_0.1_"
                "actor_lr_0.0001_critic_lr_0.001_batch_512_step_per_"
                "epoch_40000_kwargs_{}_seed_0/epoch_id_40/policy.pth")
HJconfig_path = "/home/clown2/Desktop/Work/Courses/EAIS/PytorchReachability/HJconfig.yaml"

sys.path.append(parent_dir)
sys.path.append(dreamer_dir)
print(sys.path)

import models
import tools
import ruamel.yaml as yaml
from PyHJ.exploration import GaussianNoise
from PyHJ.utils.net.common import Net
from PyHJ.utils.net.continuous import Actor, Critic
from PyHJ.policy import avoid_DDPGPolicy_annealing as DDPGPolicy
import pathlib

def recursive_update(base, update):
    for key, value in update.items():
        if isinstance(value, dict) and key in base:
            recursive_update(base[key], value)
        else:
            base[key] = value

def get_args():
    yml = yaml.YAML(typ="safe", pure=True)
    configs = yml.load((pathlib.Path(sys.argv[0]).parent.parent.parent.parent.parent / HJconfig_path).read_text())
    name_list = ["defaults"]
    defaults = {}

    for name in name_list:
        recursive_update(defaults, configs[name])
    parser = argparse.ArgumentParser()
    for key, value in sorted(defaults.items(), key=lambda x: x[0]):
        arg_type = tools.args_type(value)
        parser.add_argument(f"--{key}", type=arg_type, default=arg_type(value))
    final_config = parser.parse_args([])
    return final_config

args=get_args()
config=args

# set up the environment spaces
image_size = config.size[0] #128
img_obs_space = gym.spaces.Box(low=0, high=255, shape=(image_size, image_size, 3), dtype=np.uint8)
obs_space = gym.spaces.Box(low=0, high=1, shape=(2,), dtype=np.float32)
high = np.array([1.1, 1.1, 2*np.pi,])
low = np.array([-1.1, -1.1, 0.,])
gt_observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)
bool_space = gym.spaces.Box(low=False, high=True, shape=(), dtype=bool)
observation_space = gym.spaces.Dict({'obs_state': obs_space,
                                     'image': img_obs_space,
                                     'state': gt_observation_space,})
u_max = 1.25
action_space = gym.spaces.Box(low=-u_max, high=u_max, shape=(1,), dtype=np.float32)
config.num_actions = action_space.n if hasattr(action_space, "n") else action_space.shape[0]

# load 
config.eval_state_mean = True
wm = models.WorldModel(observation_space, action_space, 0, config)
checkpoint = torch.load(ckpt_path)
state_dict = {k[14:]:v for k,v in checkpoint['agent_state_dict'].items() if '_wm' in k}
wm.load_state_dict(state_dict)

args.state_shape = (1,1,544,)
args.action_shape = args.action1_shape = (1,)
args.max_action = args.max_action1 = u_max

# seed
np.random.seed(args.seed)
torch.manual_seed(args.seed)

activation_map = {
    'ReLU': torch.nn.ReLU,
    'Tanh': torch.nn.Tanh,
    'Sigmoid': torch.nn.Sigmoid,
    'SiLU': torch.nn.SiLU
}
actor_activation = activation_map.get(args.actor_activation)
critic_activation = activation_map.get(args.critic_activation)

assert args.critic_net is not None, "Please provide critic_net!"
critic_net = Net(
    args.state_shape,
    args.action_shape,
    hidden_sizes=args.critic_net,
    activation=critic_activation,
    concat=True,
    device=args.device
    )

critic = Critic(critic_net, device=args.device).to(args.device)
critic_optim = torch.optim.Adam(critic.parameters(), lr=args.critic_lr)

print("DDPG under the Avoid annealed Bellman equation with no Disturbance has been loaded!")

actor1_net = Net(args.state_shape, hidden_sizes=args.control_net, activation=actor_activation, device=args.device)
actor1 = Actor(actor1_net, args.action1_shape, max_action=args.max_action1, device=args.device).to(args.device)
actor1_optim = torch.optim.Adam(actor1.parameters(), lr=args.actor_lr)

policy = DDPGPolicy(
    critic,
    critic_optim,
    tau=args.tau,
    gamma=args.gamma_lcrl,
    exploration_noise=GaussianNoise(sigma=args.exploration_noise),
    reward_normalization=args.rew_norm,
    estimation_step=args.n_step,
    action_space=action_space,
    actor1=actor1,
    actor1_optim=actor1_optim,
    actor_gradient_steps=args.actor_gradient_steps,
    )

# load policy
policy.load_state_dict(torch.load(policy_path))


['/home/clown2/Desktop/Work/Courses/EAIS', '/home/clown2/anaconda3/envs/safety_rl/lib/python37.zip', '/home/clown2/anaconda3/envs/safety_rl/lib/python3.7', '/home/clown2/anaconda3/envs/safety_rl/lib/python3.7/lib-dynload', '', '/home/clown2/anaconda3/envs/safety_rl/lib/python3.7/site-packages', '/home/clown2/Desktop/Work/Courses/EAIS/PytorchReachability', '/home/clown2/anaconda3/envs/safety_rl/lib/python3.7/site-packages/IPython/extensions', '/home/clown2/.ipython', '/home/clown2/Desktop/Work/Courses/EAIS/', '/home/clown2/Desktop/Work/Courses/EAIS/eais_hw2/dreamerv3-torch']


/home/clown2/anaconda3/envs/safety_rl/lib/python3.7/site-packages/gym/logger.py:30: UserWarning: WARN: Box bound precision lowered by casting to float32
  warnings.warn(colorize('%s: %s'%('WARN', msg % args), 'yellow'))


Encoder CNN shapes: {'image': (128, 128, 3)}
Encoder MLP shapes: {'obs_state': (2,)}
Decoder CNN shapes: {'image': (128, 128, 3)}
Decoder MLP shapes: {'obs_state': (2,)}
decoder
Optimizer model_opt has 27133189 variables.
DDPG under the Avoid annealed Bellman equation with no Disturbance has been loaded!


<All keys matched successfully>

In [15]:
from PyHJ.data import Batch

def find_a(state):
    tmp_obs = np.array(state).reshape(state.shape[0],state.shape[-1])
    tmp_batch = Batch(obs = tmp_obs, info = Batch())
    tmp = policy(tmp_batch, model = "actor_old").act
    act = policy.map_action(tmp).cpu().detach().numpy().flatten()
    return act

def evaluate_V(state):
    tmp_obs = np.array(state).reshape(state.shape[0],state.shape[-1])
    tmp_batch = Batch(obs = tmp_obs, info = Batch())
    tmp = policy.critic(tmp_batch.obs, policy(tmp_batch, model="actor_old").act)
    return tmp.cpu().detach().numpy().flatten()

In [18]:
s_curr = np.array([[0., 0., 0.],[0.1, 0., 0.],[0.,0.,0.]])
s_prev = inv_dynamincs_onestep(s_curr, model_params)
# print(sp)
data_point = state_to_data(s_prev, model_params)
# print(dat)
traj = single_data_dubins(data_point)

In [22]:
s_curr = np.array([[0., 0., 0.],[0.1, 0., 0.]])
s_prev = inv_dynamincs_onestep(s_curr, model_params)
data_pts = state_to_data(s_prev, model_params)
len(data_pts)

2

In [ ]:
def dupe(traj):
    for k,v in traj.items():
        traj[k] = np.repeat(v, 2, axis=0)

dupe(traj)

In [19]:
bs = traj['state'].shape[0]
action = torch.zeros((bs,1,1), device='cuda:0')
is_first = torch.ones((bs,1), device='cuda:0')

wm.to(config.device)
idata = wm.preprocess(traj)
latent,_ = wm.dynamics.observe(wm.encoder(idata), action, is_first)
latent['stoch'] = latent['mean']
for k, v in latent.items(): latent[k] = v[:, [-1]]
feat = wm.dynamics.get_feat(latent).detach().cpu().numpy() 
value = evaluate_V(feat)
act = find_a(feat)
stat = idata['privileged_state'][0,0].cpu()
print("state: ", stat)
print("safe action: ", act)
print("safe value: ", value)


state:  tensor([-0.0500,  0.0000,  0.0000])
safe action:  [1.2499985 1.2499994 1.2499985]
safe value:  [-0.25974256 -0.29748243 -0.25974256]
